In [36]:
import torch
import pickle
import argparse
from tqdm import tqdm
import numpy as np
import pandas as pd
import os
from torch.nn.utils.rnn import pad_sequence
from terrarium.models import get_model, load_model
from terrarium.dataloaders import get_dataloader

In [37]:
model_name = 'cat_v1'
ckpt_path = 'out/cat_v1/7M/7M_1.0/ckpt.pt'
data_dir = 'data/v_32768'
dl_name = 'cat_inference'
batch_size = 10
max_seq_len = 20
total_sequences = 40
seed_lengths = list(range(0, 501, 10))
data_prefix = 'val'
summary_only = False
out_dir = './'
device = 'cpu'

In [38]:
model = load_model(model_name,ckpt_path,device_override='cpu').to(device)
checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False) # always first to cpu

In [39]:
with open(f"{data_dir}/tok.pkl",'rb') as f:
    tok = pickle.load(f)
sos_id = tok.encode('<|sos|>')
dl = get_dataloader(dl_name,batch_size,max_seq_len,f"{data_dir}/{data_prefix}_token_ids.bin",f"{data_dir}/{data_prefix}_attr_ids.bin",sos_id,device)

In [40]:
# check if dataset has enough sequences
if len(dl.split_locs)-1 < total_sequences:
    total_sequences = len(dl.split_locs)-1
    print(f'Warning: requested total_sequences reduced to {total_sequences} due to data size.')
n_batches = total_sequences // batch_size

In [41]:
# create the holder for results
results = {
    s: {
        'true_attr': [],
        'sampled_attr': [],
        'top1_attr': [],
        'cross_entropy_loss_attr': []
    }
    for s in seed_lengths
}

In [42]:
# --- run the inference over batches --- #
for b in tqdm(range(n_batches)):
    x,y,attr = dl.next_batch()
    attrs = torch.tensor([attri[0].item() for attri in attr]).to(device)

    # pad and assemble x and y
    x_pad = torch.zeros((len(x), max_seq_len), dtype=torch.long)
    y_pad = torch.zeros((len(y), max_seq_len), dtype=torch.long)
    seq_lengths = torch.zeros((len(x),), dtype=torch.long)
    for i, (xi, yi) in enumerate(zip(x, y)):
        L = len(xi)
        seq_lengths[i] = L
        x_pad[i, :L] = xi
        y_pad[i, :L] = yi
        
    x_pad,y_pad,seq_lengths = x_pad.to(device),y_pad.to(device),seq_lengths.to(device)
    
    # forward pass
    with torch.no_grad(), torch.autocast(device_type=device, dtype=torch.bfloat16):
        _,attr_logits,_,_ = model.forward(x_pad,targets=y_pad) # (B,T,A)
    B,T,A = attr_logits.shape
    cond_prob_obs = torch.softmax(attr_logits,dim=-1)

    n_minus_1_idx = (seq_lengths - 2).clamp_min(0)  # B,
    sampled_attr = torch.multinomial(cond_prob_obs.view(-1,A),num_samples=1).view(B,T) # B,T
    top1_attr = torch.argmax(cond_prob_obs,dim=-1) # B,T
    nll_attr = -torch.log(cond_prob_obs.clamp_min(1e-9)) # B,T

    for s in seed_lengths:
        rows_to_use = np.where(seq_lengths > s)[0]
        if len(rows_to_use) == 0:
            continue
        results[s]['true_attr'].append(attrs[rows_to_use].tolist())
        results[s]['sampled_attr'].append(sampled_attr[rows_to_use, s-2].tolist())
        results[s]['top1_attr'].append(top1_attr[rows_to_use, s-2].tolist())
        results[s]['cross_entropy_loss_attr'].append(nll_attr[rows_to_use, s-2, attrs[rows_to_use]].tolist())

100%|██████████| 4/4 [00:00<00:00, 46.53it/s]


In [43]:
# ---- data post processing ---- #
for s in seed_lengths:
    results[s]['true_attr'] = [item for sublist in results[s]['true_attr'] for item in sublist]
    results[s]['sampled_attr'] = [item for sublist in results[s]['sampled_attr'] for item in sublist]
    results[s]['top1_attr'] = [item for sublist in results[s]['top1_attr'] for item in sublist]
    results[s]['cross_entropy_loss_attr'] = [item for sublist in results[s]['cross_entropy_loss_attr'] for item in sublist]

In [44]:
results

{0: {'true_attr': [5,
   5,
   5,
   5,
   5,
   5,
   5,
   1,
   5,
   5,
   5,
   5,
   3,
   5,
   5,
   4,
   5,
   5,
   5,
   5,
   5,
   5,
   5,
   5,
   3,
   5,
   5,
   5,
   5,
   3,
   5,
   5,
   5,
   3,
   5,
   5,
   4,
   5,
   1,
   4],
  'sampled_attr': [0,
   2,
   2,
   5,
   0,
   1,
   4,
   3,
   5,
   0,
   1,
   0,
   5,
   2,
   5,
   5,
   3,
   5,
   1,
   4,
   3,
   2,
   1,
   2,
   4,
   5,
   0,
   2,
   0,
   0,
   5,
   4,
   2,
   3,
   4,
   1,
   1,
   3,
   4,
   2],
  'top1_attr': [2,
   2,
   1,
   5,
   2,
   4,
   4,
   4,
   2,
   2,
   4,
   4,
   4,
   2,
   5,
   4,
   2,
   4,
   2,
   4,
   2,
   4,
   2,
   2,
   4,
   5,
   2,
   2,
   4,
   1,
   4,
   4,
   0,
   3,
   4,
   2,
   2,
   2,
   4,
   2],
  'cross_entropy_loss_attr': [2.308802366256714,
   2.3082823753356934,
   2.3343677520751953,
   0.0013470181729644537,
   2.374732255935669,
   2.329939126968384,
   2.360558271408081,
   1.8203245401382446,
   2.2060132026672363,

In [45]:
# now to df for csv output, making seed length a col so can contatenate all into one file
all_dfs = []
for s in seed_lengths:
    df = pd.DataFrame({
        'seed_length': [s]*len(results[s]['true_attr']),
        'true_attr': results[s]['true_attr'],
        'sampled_attr': results[s]['sampled_attr'],
        'top1_attr': results[s]['top1_attr'],
        'cross_entropy_loss_attr': results[s]['cross_entropy_loss_attr']
    })
    all_dfs.append(df)

In [46]:
# concatenate all dataframes
final_df = pd.concat(all_dfs, ignore_index=True)
# final_df.to_csv('./out/online_classification_results.csv', index=False) # not sure want to save, could be massive
if not summary_only:
    final_df.to_csv(os.path.join(out_dir,'online_classification_results.csv'), index=False)

In [48]:
# overall summary by seed length
df_overall_summary = final_df.groupby('seed_length').agg(
    n_attrs=('true_attr', 'count'),
    sampled_accuracy=('sampled_attr', lambda x: np.mean(x == final_df.loc[x.index, 'true_attr'])),
    top1_accuracy=('top1_attr', lambda x: np.mean(x == final_df.loc[x.index, 'true_attr'])),
    cross_entropy_mean=('cross_entropy_loss_attr', 'mean')
)

In [50]:
# summary by seed and true rating
df_stratified_summary = (
    final_df
    .groupby(['seed_length', 'true_attr'])
    .agg(
        n_attrs=('true_attr', 'count'),
        sampled_accuracy=('sampled_attr', lambda x: np.mean(x == final_df.loc[x.index, 'true_attr'])),
        top1_accuracy=('top1_attr', lambda x: np.mean(x == final_df.loc[x.index, 'true_attr'])),
        cross_entropy_mean=('cross_entropy_loss_attr', 'mean')
    )
    .reset_index()  # make seed_length, true_attrs into columns again
)

In [51]:
df_overall_summary.to_csv(os.path.join(out_dir,'online_classification_overall_summary.csv'),index=False)
df_stratified_summary.to_csv(os.path.join(out_dir,'online_classification_stratified_summary.csv'),index=False)